# 06 - Plane Integration Demo

Цель: показать подключение к Plane и пример комментария COMPASS AI.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)


Project root: /Users/andrey/Documents/projects/COMPASS-AI


In [2]:
from dotenv import load_dotenv

from src.integration.plane_client import PlaneClient

load_dotenv(ROOT / ".env")

try:
    client = PlaneClient.from_env()
    print("Plane API health:", client.api_healthcheck())

    projects = client.list_projects()
    projects_df = pd.DataFrame(projects)

    display(projects_df[["id", "name", "identifier"]].head())
except Exception as exc:
    print("Plane is not available or .env is not configured:", exc)
    projects_df = pd.DataFrame()


Plane is not available or .env is not configured: type object 'PlaneClient' has no attribute 'from_env'


In [3]:
if not projects_df.empty:
    project_id = projects_df.iloc[0]["id"]
    work_items = client.list_work_items(project_id)
    work_items_df = pd.DataFrame(work_items)

    print("project_id:", project_id)
    display(work_items_df[["id", "name", "priority"]].head())
else:
    project_id = None
    work_items_df = pd.DataFrame()


In [4]:
from src.agents.orchestrator import recommend_synthetic_task

recommendation = recommend_synthetic_task(
    "TASK-0001",
    mode="balanced_workload",
    top_k=3,
    use_llm=False,
)

display(pd.DataFrame(recommendation["top_candidates"]))


,rank,employee_id,plane_user_id,name,role,grade,score,success_probability,factors,reasons,risks,source
0,1,EMP-014,nan,Никита Егоров,backend_developer,senior,0.956596,0.927648,"{'skill_match': 1.0, 'growth_match': 0.0, 'spe...","[ML model scored this candidate using task, em...",[],task_employee_matching_net
1,2,EMP-011,nan,Полина Васильева,backend_developer,senior,0.956551,0.930723,"{'skill_match': 1.0, 'growth_match': 0.0, 'spe...","[ML model scored this candidate using task, em...",[],task_employee_matching_net
2,3,EMP-010,nan,Сергей Павлов,backend_developer,middle,0.925464,0.884636,"{'skill_match': 1.0, 'growth_match': 0.2, 'spe...","[ML model scored this candidate using task, em...",[],task_employee_matching_net


In [5]:
from src.integration.plane_comment_formatter import (
    format_plane_recommendation_comment,
)

comment = format_plane_recommendation_comment(recommendation)
print(comment)


## COMPASS AI Recommendation

**Task:** Добавить endpoint для статистики команды — интеграции с Plane
**Task type:** `backend_feature`
**Mode:** `balanced_workload`
**Recommended assignee:** Никита Егоров (backend_developer, senior)
**Score:** `0.9566`
**Source:** `agentic_pipeline`

### Top candidates

1. **Никита Егоров (backend_developer, senior)** — score=0.9566, success_probability=0.9276, source=task_employee_matching_net
   - factors: skill=1.0000, risk=0.0668, workload=0.4730, growth=0.0000, quality=0.8180, deadline_reliability=0.8230
   - reasons: ML model scored this candidate using task, employee and pair features.; Mode adjustment applied: balanced_workload.
   - risks: no major risks detected
2. **Полина Васильева (backend_developer, senior)** — score=0.9566, success_probability=0.9307, source=task_employee_matching_net
   - factors: skill=1.0000, risk=0.0668, workload=0.5250, growth=0.0000, quality=1.0000, deadline_reliability=0.9510
   - reasons: ML model scored this can

In [6]:
WRITE_BACK = False

if WRITE_BACK and project_id and not work_items_df.empty:
    work_item_id = work_items_df.iloc[0]["id"]
    print("Would write comment to:", project_id, work_item_id)
else:
    print("WRITE_BACK disabled. This notebook is safe by default.")


WRITE_BACK disabled. This notebook is safe by default.
